<a href="https://colab.research.google.com/github/tanvir009/Ios-Project/blob/main/CLIP_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#new


In [ ]:
!pip install open-clip-torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00


In [ ]:
!pip install torchinfo

# CLIP Image Encoder

In [ ]:
import torch
from open_clip import create_model_and_transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, _, preprocess = create_model_and_transforms("ViT-B-16",
                                                                pretrained="openai",
                                                                device=device,
                                                                precision="fp32")

random_tensor = torch.rand(1, 3, 224, 224)

# tokens = clip_model.visual.forward_features(random_tensor)

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


In [ ]:
random_tensor.shape

torch.Size([1, 3, 224, 224])

In [ ]:
clip_model.visual(random_tensor.to(device)).shape

torch.Size([1, 512])

In [ ]:
from torchinfo import summary

summary(model = clip_model,
        input_size = (1, 3, 224, 224),
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"])

Layer (type (var_name))                                 Input Shape          Output Shape         Param #              Trainable
CLIP (CLIP)                                             [1, 3, 224, 224]     [1, 512]             63,428,097           True
├─VisionTransformer (visual)                            [1, 3, 224, 224]     [1, 512]             545,280              True
│    └─Conv2d (conv1)                                   [1, 3, 224, 224]     [1, 768, 14, 14]     589,824              True
│    └─Identity (patch_dropout)                         [1, 197, 768]        [1, 197, 768]        --                   --
│    └─LayerNorm (ln_pre)                               [1, 197, 768]        [1, 197, 768]        1,536                True
│    └─Transformer (transformer)                        [1, 197, 768]        [1, 197, 768]        --                   True
│    │    └─ModuleList (resblocks)                      --                   --                   85,054,464           True
│    

In [ ]:
import torch

# Create a random input image (batch=1, 3 channels, 224x224)
random_tensor = torch.rand(1, 3, 224, 224).to(device)

# Forward pass through the vision encoder (up to ln_post)
with torch.no_grad():
    x = clip_model.visual.conv1(random_tensor)   # patchify
    x = x.reshape(x.shape[0], x.shape[1], -1)    # flatten patches
    x = x.permute(0, 2, 1)                       # (batch, seq, dim)
    x = clip_model.visual.ln_pre(x)

    for block in clip_model.visual.transformer.resblocks:
        x = block(x)

    ln_post_out = clip_model.visual.ln_post(x)

print("ln_post output shape:", ln_post_out.shape)


ln_post output shape: torch.Size([1, 196, 768])


In [ ]:
ln_post_out = None

def hook_fn(module, input, output):
    global ln_post_out
    ln_post_out = output

# Register hook
hook = clip_model.visual.ln_post.register_forward_hook(hook_fn)

# Run model
with torch.no_grad():
    _ = clip_model.visual(random_tensor.to(device))

hook.remove()

print("ln_post output shape:", ln_post_out.shape)


ln_post output shape: torch.Size([1, 197, 768])


In [ ]:
import torch

# Example text tokens (tokenized beforehand)
# Let's simulate random token IDs just for shape testing
text_tokens = torch.randint(0, clip_model.token_embedding.num_embeddings, (1, 10)).to(device)

with torch.no_grad():
    # Token embedding
    x = clip_model.token_embedding(text_tokens)  # (batch, seq_len, dim)

    # Transformer
    for block in clip_model.transformer.resblocks:
        x = block(x)

    # Final layer norm
    ln_final_out = clip_model.ln_final(x)

print("ln_final output shape:", ln_final_out.shape)


ln_final output shape: torch.Size([1, 10, 512])


In [ ]:
# tokens = clip_model.visual.forward_intermediate(random_tensor)

In [ ]:
##### FOR ONLY THE 12TH LAYER ####
# def forward_features(self, x):
#     x = self.conv1(x)  # patchify
#     x = x.reshape(x.shape[0], x.shape[1], -1)
#     x = x.permute(0, 2, 1)
#     cls_token = self.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device)
#     x = torch.cat([cls_token, x], dim=1)
#     x = x + self.positional_embedding.to(x.dtype)
#     x = self.ln_pre(x)

#     for blk in self.transformer.resblocks:
#         x = blk(x)
#     return x  # [batch, seq_len, width]


##### FOR ALL THE LAYERS #####
def forward_features(self, x):
    all_layers = []   # to collect embeddings from every encoder

    x = self.conv1(x)  # patchify
    x = x.reshape(x.shape[0], x.shape[1], -1)
    x = x.permute(0, 2, 1)
    cls_token = self.class_embedding.to(x.dtype) + torch.zeros(
        x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device
    )
    x = torch.cat([cls_token, x], dim=1)
    x = x + self.positional_embedding.to(x.dtype)
    x = self.ln_pre(x)

    for blk in self.transformer.resblocks:
        x = blk(x)
        all_layers.append(x)  # save after each block

    return all_layers  # list of [batch, seq_len, width]


# attach to model
import types
clip_model.visual.forward_features = types.MethodType(forward_features, clip_model.visual)

# now this works (For 12th encoder layer):
# tokens = clip_model.visual.forward_features(random_tensor.to(device))
# print(tokens.shape)  # (1, 197, 768) for ViT-B/16, 224x224 input




# For all layers
tokens_per_layer = clip_model.visual.forward_features(random_tensor.to(device))
print(f"There are {len(tokens_per_layer)} layers")        # 12 layers
print(f"Shaper of tensor after Layer-1: {tokens_per_layer[0].shape}")    # (1, 197, 768)
print(f"Shaper of tensor after Layer-2: {tokens_per_layer[1].shape}")   # (1, 197, 768)

There are 12 layers
Shaper of tensor after Layer-1: torch.Size([1, 197, 768])
Shaper of tensor after Layer-2: torch.Size([1, 197, 768])


In [ ]:
print(clip_model)

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

In [ ]:
# Hook function
def print_shape_hook(module, input, output):
    try:
        inp_shape = input[0].shape if isinstance(input, (list, tuple)) else input.shape
        out_shape = output.shape if hasattr(output, "shape") else None
        print(f"{module.__class__.__name__}: input {inp_shape}, output {out_shape}")
    except:
        pass  # Some modules may not have clean shapes (like containers)

# Register hooks on all submodules
for name, layer in clip_model.named_modules():
    if len(list(layer.children())) == 0:  # only leaf layers
        layer.register_forward_hook(print_shape_hook)

# Dummy input (batch_size=1, 3x224x224 image)
x = torch.randn(1, 3, 224, 224).to(device)

# Forward pass through visual encoder
with torch.no_grad():
    _ = clip_model.visual(x)

Conv2d: input torch.Size([1, 3, 224, 224]), output torch.Size([1, 768, 14, 14])
Identity: input torch.Size([1, 197, 768]), output torch.Size([1, 197, 768])
LayerNorm: input torch.Size([1, 197, 768]), output torch.Size([1, 197, 768])
LayerNorm: input torch.Size([1, 197, 768]), output torch.Size([1, 197, 768])
Identity: input torch.Size([1, 197, 768]), output torch.Size([1, 197, 768])
LayerNorm: input torch.Size([1, 197, 768]), output torch.Size([1, 197, 768])
Linear: input torch.Size([1, 197, 768]), output torch.Size([1, 197, 3072])
GELU: input torch.Size([1, 197, 3072]), output torch.Size([1, 197, 3072])
Linear: input torch.Size([1, 197, 3072]), output torch.Size([1, 197, 768])
Identity: input torch.Size([1, 197, 768]), output torch.Size([1, 197, 768])
LayerNorm: input torch.Size([1, 197, 768]), output torch.Size([1, 197, 768])
Identity: input torch.Size([1, 197, 768]), output torch.Size([1, 197, 768])
LayerNorm: input torch.Size([1, 197, 768]), output torch.Size([1, 197, 768])
Linear:

In [ ]:
# # tokens: [B, 197, 768]
# patch_tokens = tokens[:, 1:, :]  # exclude CLS
# cls_token = tokens[:, 0, :]

# # proj is a Parameter: [768, 512]
# proj_weight = clip_model.visual.proj  # shape [768, 512]

# # manual projection
# patch_tokens_512 = patch_tokens @ proj_weight   # [B, 196, 512]
# cls_token_512 = cls_token @ proj_weight         # [B, 512]

# print(patch_tokens_512.shape, cls_token_512.shape)

torch.Size([1, 196, 512]) torch.Size([1, 512])


In [ ]:
class Dummy_model:
  def __init__(self):
    pass
  def forward(self, x):
    pass

dummy_model = Dummy_model()

In [ ]:
from torchinfo import summary

summary(model = dummy_model,
        input_size = (1, 3, 224, 224),
        col_names = ["input_size", "output_size", "num_params", "trainable"],
        col_width = 20,
        row_settings = ["var_names"])

AttributeError: 'Dummy_model' object has no attribute 'parameters'